In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# Загрузка данных
df = pd.read_csv('creditcard.csv')

# Разделение на признаки и цель
y = df['Class']
X = df.drop(columns=['Class'])

In [2]:
# Сначала отделяем Test (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Затем из оставшегося отделяем Val (25% от 80% = 20% от всех)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

# Сброс индексов (чтобы не было путаницы)
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

print(f"Train: {X_train.shape}, фрод: {y_train.sum()}")
print(f"Val:   {X_val.shape}, фрод: {y_val.sum()}")
print(f"Test:  {X_test.shape}, фрод: {y_test.sum()}")

Train: (170883, 30), фрод: 295
Val:   (56962, 30), фрод: 99
Test:  (56962, 30), фрод: 98


In [3]:
# Обучаем scaler ТОЛЬКО на train, чтобы не было утечки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Масштабирование выполнено.")

Масштабирование выполнено.


In [4]:
# Обучаем на train
model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Модель обучена.")

Модель обучена.


In [5]:
y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
y_val_pred_05 = (y_val_proba >= 0.5).astype(int)

print("=== LOGISTIC REGRESSION (VAL, threshold=0.5) ===")
print(f"PR AUC   : {average_precision_score(y_val, y_val_proba):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_05):.4f}")
print(f"Recall   : {recall_score(y_val, y_val_pred_05):.4f}")
print(f"F1       : {f1_score(y_val, y_val_pred_05):.4f}")

=== LOGISTIC REGRESSION (VAL, threshold=0.5) ===
PR AUC   : 0.7091
Precision: 0.8254
Recall   : 0.5253
F1       : 0.6420


In [6]:
print(f"\n{'Threshold':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 45)

best_threshold = 0
best_f1 = 0

for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_val_pred = (y_val_proba >= t).astype(int)
    p = precision_score(y_val, y_val_pred, zero_division=0)
    r = recall_score(y_val, y_val_pred, zero_division=0)
    f = f1_score(y_val, y_val_pred, zero_division=0)
    
    print(f"{t:<12.1f} {p:>10.4f} {r:>10.4f} {f:>10.4f}")
    
    if f > best_f1:
        best_f1 = f
        best_threshold = t

print(f"\nЛучший порог: {best_threshold} (F1 = {best_f1:.4f})")


Threshold     Precision     Recall         F1
---------------------------------------------
0.1              0.8140     0.7071     0.7568
0.2              0.8205     0.6465     0.7232
0.3              0.8289     0.6364     0.7200
0.4              0.8169     0.5859     0.6824
0.5              0.8254     0.5253     0.6420
0.6              0.8475     0.5051     0.6329
0.7              0.8421     0.4848     0.6154
0.8              0.8393     0.4747     0.6065
0.9              0.8333     0.4545     0.5882

Лучший порог: 0.1 (F1 = 0.7568)


In [7]:
# 1. Масштабируем train+val заново
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(X_train_full)
X_test_scaled_final = scaler_full.transform(X_test)

# 2. Обучаем модель на train+val
model_full = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
model_full.fit(X_train_scaled_full, y_train_full)

# 3. Предсказания на тесте с ВЫБРАННЫМ порогом
y_test_proba = model_full.predict_proba(X_test_scaled_final)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

print(f"=== LOGISTIC REGRESSION (FINAL, threshold={best_threshold}) ===")
print(f"PR AUC   : {average_precision_score(y_test, y_test_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred):.4f}")
print(f"F1       : {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_test_proba):.4f}")

=== LOGISTIC REGRESSION (FINAL, threshold=0.1) ===
PR AUC   : 0.7414
Precision: 0.7009
Recall   : 0.7653
F1       : 0.7317
ROC AUC  : 0.9605
